# FoodMkt_R — Practice Skeleton

**Short name:** `FoodMkt_R` (GitHub-friendly). Food-industry adaptation of Packt *Practical Data Science Cookbook* 2e, Ch. 4 (`StockMkt_R`).

**Goal.** Acquire a Finviz-style snapshot of **food / CPG / restaurant / agribusiness** names, clean messy numerics, hunt the luxury-chocolate outlier that warps sector means, build a 10-flag **relative valuation index** against food-industry peers, screen a short US list, and overlay 50/200-day moving averages.

**Not investment advice. Not a procurement award.** Same teaching-pipeline caveat as the stock chapter.

**Offline data.** Use `data/foodviz.csv` and `data/food_historical_prices.csv`. Live vendor URLs are optional comments only.

**Companion files.** Solution notebook · `FoodMkt_R_Cheatsheet.docx` · reusable template · 1-page report · memo · strategy guide · `foodmkt_r_flowchart.png`.


## 0. Setup

Load `ggplot2`, `plyr` (or `dplyr`), `reshape2` (or `tidyr`), `zoo`.


In [ ]:
# YOUR CODE HERE
# library(ggplot2)
# library(plyr)
# library(reshape2)
# library(zoo)
# theme_set(theme_minimal())

## 1. Acquire the food snapshot

### Task 1–3
1. Read `data/foodviz.csv` into `foodviz`. Keep strings as strings (`stringsAsFactors = FALSE`, `check.names = FALSE`).
2. `head(foodviz[, 1:6])` and `dim(foodviz)`.
3. How many unique `Sector` values? Which sector has the most names?

Optional live pattern (do not depend on it):

```r
url_to_open <- sprintf("http://finviz.com/export.ashx?v=152&c=%s", paste(0:68, collapse = ","))
```


In [ ]:
# Task 1
foodviz <- # YOUR CODE HERE

# Tasks 2–3
# YOUR CODE HERE

## 2. Summarize fields and learn the food vocabulary

### Task 4–7
4. `summary(foodviz[, 1:6])` and `sort(table(foodviz$Sector), decreasing = TRUE)`.
5. Identifying fields: Ticker, Company, Sector, Industry, Country.
6. One sentence each for **Price, P/E, PEG, Debt/Equity, Beta, Gross Margin, Food Cost %, SSS %**.
7. Why is *Packaged Foods* coarser than *Cereal & Breakfast*? Give one name from this file.


In [ ]:
# Tasks 4–7
# YOUR CODE HERE

## 3. Clean numerics (`clean_numeric`)

Growth, ownership, gross margin, food cost and SSS arrive with `%`. Volume has commas.

### Task 8–10
8. `clean_numeric <- function(s) { s <- gsub("%|\\$|,|\\)|\\(", "", s); as.numeric(s) }`
9. Apply to every column after the six identifiers.
10. `names(foodviz) <- make.names(names(foodviz))` then `str()`. Confirm `Price`, `P.E`, `Gross.Margin`, `Food.Cost..` are numeric.

**Column-index warning:** do not hard-code `7:68` on a new export.


In [ ]:
# Tasks 8–10
clean_numeric <- function(s) {
  # YOUR CODE HERE
}

# foodviz <- cbind(...)
# YOUR CODE HERE

## 4. Explore the price distribution

### Task 11–14
11. `hist(foodviz$Price, breaks = 100)` — why is this chart useless?
12. Cap: `hist(foodviz$Price[foodviz$Price < 150], breaks = 100)`.
13. Sector means with `aggregate(Price ~ Sector, data = foodviz, FUN = mean)` and a `ggplot` bar.
14. Which sector looks expensive, and what do you suspect before drilling down?


In [ ]:
# Tasks 11–14
# YOUR CODE HERE

## 5. Drill Confectionery → industry → company, then drop GODIVA

### Task 15–18
15. Industry means, then `subset(..., Sector == "Confectionery")`. Bar the industries.
16. Subset `Industry == "Luxury Chocolate"` and bar companies (rotate x labels).
17. Name the outlier ticker and its price.
18. `foodviz <- subset(foodviz, Ticker != "GODIVA")` and recompute sector means. Did Confectionery come back to earth?


In [ ]:
# Tasks 15–18
# YOUR CODE HERE

## 6. Relative valuation — sector and industry averages

Compare each food name to *similar food names*, not to a DCF of cocoa beans.

### Task 19–23
19. `sector_avg <- melt(foodviz, id = "Sector")` then keep `Price`, `P.E`, `PEG`, `P.S`, `P.B`.
20. `na.omit`, numeric `value`, `dcast(..., Sector ~ variable, mean)`. Rename `SAvg*`.
21. Repeat at industry grain → `IAvg*`.
22. `merge` both tables back.
23. Why did `nrow` drop? Acceptable for a *screen*?


In [ ]:
# Tasks 19–23
# YOUR CODE HERE

## 7. Ten under-average flags → RelValIndex

### Task 24–27
24. Ten 0/1 columns: sector and industry × Price, P/E, PEG, P/S, P/B.
25. Flip to 1 when the name is **strictly below** the matching food-peer average.
26. `RelValIndex <- rowSums(...)` (0–10).
27. `potentially_undervalued <- subset(foodviz, RelValIndex >= 8)`.


In [ ]:
# Tasks 24–27
# YOUR CODE HERE

## 8. Screen a target list and inspect food-name histories

Example filters (edit later):

- Country == "USA"
- Price in (20, 100)
- Volume > 10,000
- EPS (ttm) > 0 and both growth fields > 0
- Total Debt/Equity < 1
- Beta < 1.5
- Institutional Ownership < 30
- RelValIndex >= 8

### Task 28–32
28. `target_stocks <- subset(...)`.
29. Read `data/food_historical_prices.csv`.
30. For **GIS** (or first target) compute 50- and 200-day MAs with `zoo::rollmean` and line-plot.
31. Combined AdjClose by Symbol.
32. Open-first / High-max / Low-min / Close-last bars.


In [ ]:
# Tasks 28–32
target_stocks <- # YOUR CODE HERE

hist_px <- # YOUR CODE HERE

# MA + charts
# YOUR CODE HERE

## Alternate code (same results)

Re-do **one**:

- Sector means: `aggregate` ↔ `plyr::ddply` ↔ `dplyr::summarise`
- RelVal averages: `melt`/`dcast` ↔ `tidyr::pivot_*`
- Flags: ten assignments ↔ helper / `across()`
- MAs: `zoo::rollmean` ↔ `stats::filter(rep(1/n, n), sides = 1)`


In [ ]:
# YOUR CODE HERE — pick one alternate

## More practice

1. Rebuild RelValIndex with **median** sector stats. How many names still score ≥ 8?
2. Add food-quality flags: `Gross.Margin > sector median` and `Food.Cost.. < 40` (index 0–12).
3. Correlation heatmap of daily returns among HSY / GIS / TSN / ADM / MCD.
4. Boxplots of `Gross.Margin` for Restaurants vs Grocery Retail.
5. Among restaurants with non-missing `SSS..`, who has SSS > 3?


In [ ]:
# YOUR CODE HERE — at least two practice items

## Simulation / what-if

Knobs:

- `idx_cut` — RelValIndex minimum (6, 8, 10)
- `price_lo`, `price_hi`
- `max_de`, `max_beta`, `max_inst`
- `usa_only`
- `max_foodcost` — extra food-cost cap (try 35, 50, 80)
- `noise_sd` on Price

Record `n_pass`, median Price, sector mix.


In [ ]:
idx_cut <- 8
price_lo <- 20
price_hi <- 100
max_de <- 1
max_beta <- 1.5
max_inst <- 30
usa_only <- TRUE
max_foodcost <- 80
noise_sd <- 0
set.seed(21)

# YOUR CODE HERE

## Audience rewrite

Same finding, four ways (attached audience PDFs):

*“One Swiss luxury-chocolate listing (GODIVA at $18,750) pulled Confectionery — and the all-food mean — out of line. After dropping it, sector average prices sit in a tight band. A 10-flag peer index plus a conservative US screen left a short CPG/restaurant list whose 50-day MAs can be compared with 200-day MAs.”*

1. **Expert / CPG quant**
2. **Technician / category screener**
3. **Executive / CPG CFO or investment committee**
4. **Nonspecialist / grocery shopper**


In [ ]:
# YOUR TEXT HERE

## Next

Open `FoodMkt_R_Solution.ipynb` after attempting each task. Keep `FoodMkt_R_Cheatsheet.docx` beside the skeleton.
